In [1]:
import os
os.chdir('/home/bluefox/NN/NN-Project1')

In [19]:
import keras
import tensorflow as tf
import numpy as np
from keras import layers, models
from keras.utils import text_dataset_from_directory
import spacy
import re
import pickle

In [3]:
batch_size = 32
raw_train_ds = text_dataset_from_directory(
    "data/raw/aclImdb/train",
    batch_size=batch_size,
    validation_split=0.2,
    subset="training",
    seed=1337,
)
raw_val_ds = text_dataset_from_directory(
    "data/raw/aclImdb/train",
    batch_size=batch_size,
    validation_split=0.2,
    subset="validation",
    seed=1337,
)
raw_test_ds = text_dataset_from_directory(
    "data/raw/aclImdb/test",
    batch_size=batch_size,
    seed=1337,
)

print(f"Number of batches in raw_train_ds: {raw_train_ds.cardinality()}")
print(f"Number of batches in raw_val_ds: {raw_val_ds.cardinality()}")
print(f"Number of batches in raw_test_ds: {raw_test_ds.cardinality()}")

Found 25000 files belonging to 2 classes.
Using 20000 files for training.


2026-05-17 21:48:48.161148: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Found 25000 files belonging to 2 classes.
Using 5000 files for validation.
Found 25000 files belonging to 2 classes.
Number of batches in raw_train_ds: 625
Number of batches in raw_val_ds: 157
Number of batches in raw_test_ds: 782


In [ ]:
nlp = spacy.load("en_core_web_sm")

def spacy_clean_batch(text_tensor):
    byte_array = text_tensor.numpy()
    
    if byte_array.ndim == 0:
        byte_array = np.array([byte_array])
        
    texts = [b.decode("utf-8") for b in byte_array]
    texts = [re.sub(r"<br\s*/?>", " ", t.lower()) for t in texts]
    
    cleaned_batch = []
    for doc in nlp.pipe(texts):
        tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and not token.is_space]
        cleaned_batch.append(" ".join(tokens))
        
    return np.array(cleaned_batch, dtype=object)


def tf_spacy_wrapper(text_batch, label_batch):
    clean_text_batch = tf.py_function(
        func=spacy_clean_batch, 
        inp=[text_batch], 
        Tout=tf.string
    )

    clean_text_batch = tf.reshape(clean_text_batch, [-1])
    label_batch = tf.reshape(label_batch, [-1])
    
    clean_text_batch.set_shape([None])
    label_batch.set_shape([None])
    
    return clean_text_batch, label_batch

In [6]:
max_features = 20000
embedding_dim = 128
sequence_length = 500

In [7]:
clean_train_ds = raw_train_ds.map(tf_spacy_wrapper, num_parallel_calls=tf.data.AUTOTUNE)
clean_val_ds   = raw_val_ds.map(tf_spacy_wrapper, num_parallel_calls=tf.data.AUTOTUNE)
clean_test_ds  = raw_test_ds.map(tf_spacy_wrapper, num_parallel_calls=tf.data.AUTOTUNE)

In [8]:
vectorize_layer = layers.TextVectorization(
    standardize=None,
    max_tokens=max_features,
    output_mode="int",
    output_sequence_length=sequence_length
)

In [9]:
text_ds = clean_train_ds.map(lambda x, y: x)

In [10]:
vectorize_layer.adapt(text_ds)

2026-05-17 21:52:25.419986: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [11]:
def vectorize_text(text, label):
    text = tf.expand_dims(text, [-1])
    return vectorize_layer(text), label

In [12]:
train_ds = clean_train_ds.map(vectorize_text, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

val_ds = clean_val_ds.map(vectorize_text, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

test_ds = clean_test_ds.map(vectorize_text, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

In [14]:
train_ds.save("data/processed/train_ds")

In [15]:
val_ds.save("data/processed/val_ds")

In [16]:
test_ds.save("data/processed/test_ds")

In [20]:
vocab = vectorize_layer.get_vocabulary()
with open("data/processed/vectorize_layer.pkl", "wb") as f:
    pickle.dump(vectorize_layer, f)